# AURORA OMEGA MAX V6 - Economic Gap Model

V6 is not another "pick the least mediocre model" notebook.

It tests a better economic architecture:

- infer what the market price already implies through the reverse DCF spine;
- forecast the business trajectory first: revenue growth, operating margin, ROIC, and FCF margin;
- compare forecast fundamentals against market-implied expectations;
- let a small robust head adjust the deterministic valuation spine only when the economic gap carries evidence;
- validate with the same purged rolling-origin protocol as V5.1.

The expected win is not scale. The expected win is causality discipline: price expectations versus business capacity.

## 1. Runtime and Config


In [ ]:
import os, sys, json, time, random, subprocess, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
    import torch

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, HuberRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore", category=UserWarning)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

REPO_URL = "https://github.com/tbasaure-sys/fin.git"
REPO_REF = "main"
WORKDIR = Path("/content/fin") if IN_COLAB else Path.cwd()
DRIVE_ROOT = Path("/content/drive/MyDrive/blsprime_aurora_omega") if IN_COLAB else Path("./_local_data/blsprime_aurora_omega")
PANEL_ROOT = DRIVE_ROOT / "panel"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"

NOTEBOOK_VERSION = "aurora_omega_max_v6_economic_gap_model"
ARTIFACT_NAME_PREFIX = "omega_v6_economic_gap_model"
DATA_CUTOFF_DATE = pd.Timestamp.utcnow().tz_localize(None).date().isoformat()
HORIZON_YEARS = 3
TARGET = "ann_return_3y_fwd"
SEED = 11
MIN_CORE_ROWS = 1200
MIN_TUNE_ROWS = 350
MIN_VAL_ROWS = 100
ROLLING_VAL_YEARS = list(range(2013, 2023))

for p in [DRIVE_ROOT, PANEL_ROOT, ARTIFACT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("Runtime:", {
    "python": sys.version.split()[0],
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
print("Drive:", DRIVE_ROOT)
print("Data cutoff:", DATA_CUTOFF_DATE)
print("Validation years:", ROLLING_VAL_YEARS)

## 2. Sync Repo and Imports


In [ ]:
if IN_COLAB:
    if not WORKDIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(WORKDIR)])
    subprocess.check_call(["git", "-C", str(WORKDIR), "fetch", "origin", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "checkout", REPO_REF])
    subprocess.check_call(["git", "-C", str(WORKDIR), "pull", "--ff-only", "origin", REPO_REF])

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

import importlib
import scripts.run_aurora_router_local as router
importlib.reload(router)

DEFAULT_LENS_NAMES = [
    "dcf", "roicFade", "reverseDcf", "residualIncome",
    "assetValue", "unitEconomics", "bottleneck", "realOptions", "capitalCycle",
]
try:
    from aurora_omega.data import LENS_NAMES as PACKAGE_LENS_NAMES
    LENS_NAMES = list(PACKAGE_LENS_NAMES)
except Exception:
    LENS_NAMES = DEFAULT_LENS_NAMES

print("Repo:", WORKDIR)
print("Lenses:", LENS_NAMES)

## 3. Rebuild Featured Panel and Harden Universe Filter


In [ ]:
PANEL_CANDIDATES = [
    PANEL_ROOT / "panel_autodiscover_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_cache_only_2005_2024_1500.parquet",
    PANEL_ROOT / "panel_selfcontained_2005_2024_1500.parquet",
]
panel_path = next((p for p in PANEL_CANDIDATES if p.exists() and p.stat().st_size > 0), None)
if panel_path is None:
    raise FileNotFoundError("No cached panel found. Expected one of: " + ", ".join(map(str, PANEL_CANDIDATES)))

panel = pd.read_parquet(panel_path)
print("Loaded raw panel:", panel_path, panel.shape, "tickers:", panel["ticker"].nunique())

featured = router.add_features(panel.copy())
featured = router.add_lens_predictions(featured)
featured["omega_regime"] = featured.apply(router.classify_spine_regime, axis=1)
featured["omega_primary_question"] = featured["omega_regime"].map(router.primary_question_for_regime)

expectation_keys = [
    "implied_revenue_cagr",
    "implied_terminal_ebit_margin",
    "implied_incremental_roic",
    "implied_reinvestment_rate",
    "duration_risk",
    "valuation_pressure_score",
    "current_revenue_growth_3y",
    "current_operating_margin",
    "current_roic_proxy",
    "cost_anchor",
]
expectations = featured.apply(router.reverse_dcf_expectations, axis=1)
for key in expectation_keys:
    featured[f"exp_{key}"] = [e.get(key, np.nan) for e in expectations]

featured["omega_expectations_pressure"] = featured["exp_valuation_pressure_score"]
featured["omega_feasibility_score"] = [
    router.score_expectation_feasibility(row, e).get("score", np.nan)
    for (_, row), e in zip(featured.iterrows(), expectations)
]
featured["omega_downside_anchor_score"] = [
    router.anchor_lens_checks(row, router.classify_spine_regime(row), e).get("asset_value", {}).get("score", np.nan)
    for (_, row), e in zip(featured.iterrows(), expectations)
]

MUST_EXCLUDE_PRODUCT_TICKERS = {
    "ABALX", "FNILX", "VTSAX", "VBTIX", "GBTC", "ETHE", "IBIT", "FBTC", "BITB", "ARKB",
    "SLV", "GLD", "IAU", "USO", "UNG", "SPY", "QQQ", "VOO", "VTI", "IWM", "DIA",
    "TLT", "HYG", "LQD", "BND", "SHY", "IEF", "EEM", "EFA", "XLF", "XLK", "XLE", "XLV",
}
VALID_OPERATING_CANARY_KEEP = {"CVX", "BSX", "BDX", "EQIX", "X"}


def common_operating_equity_mask(frame):
    ticker = frame["ticker"].astype(str).str.upper().str.strip()
    sector = frame.get("sector", pd.Series("Unknown", index=frame.index)).astype(str).str.lower()
    industry = frame.get("industry", pd.Series("Unknown", index=frame.index)).astype(str).str.lower()
    name = frame.get("company_name", pd.Series("", index=frame.index)).astype(str).str.lower()
    text = sector + " " + industry + " " + name
    operating_symbol = ticker.str.match(r"^[A-Z]{1,5}([.-][A-Z])?$")
    blank_sector = sector.isin(["", "unknown", "nan", "none"])
    fund_like_text = text.str.contains(
        r"mutual fund|index fund|exchange traded fund|\betf\b|closed-end|target date|money market|portfolio|trust|treasury|bond fund|income fund|municipal|variable insurance|blackrock|vanguard|fidelity|ishares|spdr",
        regex=True,
        na=False,
    )
    product_ticker = ticker.isin(MUST_EXCLUDE_PRODUCT_TICKERS)
    fund_family_share_class = ticker.str.match(r"^[A-Z]{4}X$") & fund_like_text
    valid_canary = ticker.isin(VALID_OPERATING_CANARY_KEEP)
    return operating_symbol & ((~blank_sector) | valid_canary) & (~product_ticker) & (~fund_like_text | valid_canary) & (~fund_family_share_class | valid_canary)


mask = common_operating_equity_mask(featured)
removed = featured.loc[~mask, ["ticker", "company_name", "sector", "industry"]].drop_duplicates().sort_values("ticker")
featured = featured.loc[mask].copy()

raw_ticker_set = set(panel["ticker"].astype(str).str.upper())
filtered_ticker_set = set(featured["ticker"].astype(str).str.upper())
raw_operating_canaries_present = raw_ticker_set & VALID_OPERATING_CANARY_KEEP
filter_audit = {
    "raw_rows": int(len(panel)),
    "filtered_rows": int(len(featured)),
    "raw_tickers": int(panel["ticker"].nunique()),
    "filtered_tickers": int(featured["ticker"].nunique()),
    "removed_ticker_count": int(removed["ticker"].nunique()) if len(removed) else 0,
    "must_exclude_product_survivors": sorted(filtered_ticker_set & MUST_EXCLUDE_PRODUCT_TICKERS),
    "raw_operating_canaries_present": sorted(raw_operating_canaries_present),
    "wrongly_removed_operating_canaries": sorted(raw_operating_canaries_present - filtered_ticker_set),
}
print(json.dumps(filter_audit, indent=2))
display(featured[["ticker", "year", "omega_regime", "omega_primary_question", "exp_implied_revenue_cagr", "exp_implied_terminal_ebit_margin", "omega_feasibility_score"]].head())

## 4. Future Fundamental Targets and Shared Helpers


In [ ]:
def mask_immature_forward_returns(frame, target_col=TARGET, horizon_years=HORIZON_YEARS, cutoff_date=DATA_CUTOFF_DATE):
    out = frame.copy()
    asof = pd.to_datetime(out["asof_date"], errors="coerce")
    cutoff = pd.Timestamp(cutoff_date)
    if cutoff.tzinfo is not None:
        cutoff = cutoff.tz_localize(None)
    mature_date = asof + pd.DateOffset(years=horizon_years)
    matured = mature_date.notna() & (mature_date <= cutoff)
    out[f"{target_col}_matured"] = matured
    out.loc[~matured, target_col] = np.nan
    return out


def safe_div(a, b):
    return np.where(np.abs(b) > 1e-12, a / b, np.nan)


def add_future_fundamental_targets(frame, horizon=HORIZON_YEARS):
    out = frame.copy().sort_values(["ticker", "year"])
    g = out.groupby("ticker", sort=False)
    out[f"future_revenue_{horizon}y"] = g["revenue"].shift(-horizon)
    out[f"future_operating_margin_{horizon}y"] = g["operating_margin"].shift(-horizon)
    out[f"future_roic_{horizon}y"] = g["roic_proxy"].shift(-horizon)
    out[f"future_fcf_margin_{horizon}y"] = g["fcf_margin"].shift(-horizon)
    rev_ratio = safe_div(out[f"future_revenue_{horizon}y"].astype(float), out["revenue"].astype(float))
    out[f"future_revenue_cagr_{horizon}y"] = np.where(rev_ratio > 0, np.power(rev_ratio, 1.0 / horizon) - 1.0, np.nan)
    out[f"realized_growth_gap_{horizon}y"] = out[f"future_revenue_cagr_{horizon}y"] - out["exp_implied_revenue_cagr"]
    out[f"realized_margin_gap_{horizon}y"] = out[f"future_operating_margin_{horizon}y"] - out["exp_implied_terminal_ebit_margin"]
    out[f"realized_roic_gap_{horizon}y"] = out[f"future_roic_{horizon}y"] - out["exp_implied_incremental_roic"]
    out[f"realized_fcf_delta_{horizon}y"] = out[f"future_fcf_margin_{horizon}y"] - out["fcf_margin"]
    return out


def mae_np(pred, y):
    s = pd.DataFrame({"pred": pred, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    return float(np.mean(np.abs(s["pred"] - s["y"]))) if len(s) else float("nan")


def ic_np(pred, y):
    s = pd.DataFrame({"pred": pred, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 20 or s["pred"].nunique() < 5 or s["y"].nunique() < 5:
        return float("nan")
    return float(s["pred"].rank().corr(s["y"].rank()))


def decile_spread(frame, pred_col, target_col=TARGET):
    spreads = []
    for _, sub in frame[["year", pred_col, target_col]].dropna().groupby("year"):
        if len(sub) < 80 or sub[pred_col].nunique() < 10:
            continue
        q = pd.qcut(sub[pred_col], 10, labels=False, duplicates="drop")
        if q.max() < 1:
            continue
        spreads.append(float(sub.loc[q == q.max(), target_col].mean() - sub.loc[q == q.min(), target_col].mean()))
    return float(np.mean(spreads)) if spreads else float("nan")


def prior_for_lenses(names):
    raw = []
    for n in names:
        if n == "reverseDcf":
            raw.append(0.36)
        elif n == "assetValue":
            raw.append(0.26)
        elif n == "residualIncome":
            raw.append(0.17)
        elif n == "roicFade":
            raw.append(0.08)
        elif n == "dcf":
            raw.append(0.07)
        elif n == "unitEconomics":
            raw.append(0.04)
        else:
            raw.append(0.02)
    raw = np.asarray(raw, dtype="float64")
    return raw / raw.sum()


def fit_simplex_spine(frame, lens_cols, target_col=TARGET, epochs=1600, lr=0.05, l2=0.04):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X = torch.tensor(frame[lens_cols].values.astype("float32"), device=device)
    y = torch.tensor(frame[target_col].values.astype("float32"), device=device)
    prior = torch.tensor(prior_for_lenses([c.replace("pred_", "") for c in lens_cols]).astype("float32"), device=device)
    theta = torch.zeros(len(lens_cols), device=device, requires_grad=True)
    opt = torch.optim.Adam([theta], lr=lr)
    for _ in range(epochs):
        w = torch.softmax(theta, dim=0)
        pred = X @ w
        loss = torch.mean(torch.abs(pred - y)) + l2 * torch.mean((w - prior) ** 2)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return torch.softmax(theta, dim=0).detach().cpu().numpy()


def apply_spine(frame, lens_cols, weights):
    return frame[lens_cols].values.astype("float64") @ np.asarray(weights, dtype="float64")

## 5. Economic Gap Feature Model


In [ ]:
FUNDAMENTAL_TARGETS = [
    "future_revenue_cagr_3y",
    "future_operating_margin_3y",
    "future_roic_3y",
    "future_fcf_margin_3y",
]

BASE_NUMERIC_FEATURES = [
    "revenue", "gross_margin", "operating_margin", "fcf_margin", "roic_proxy",
    "debt_assets", "capex_intensity", "revenue_growth_3y", "asset_turnover",
    "ev_to_sales", "pb", "fcf_yield", "risk_free_10y", "inflation",
    "gross_margin_year_z", "operating_margin_year_z", "fcf_margin_year_z",
    "roic_proxy_year_z", "revenue_growth_3y_year_z", "debt_assets_year_z",
    "capex_intensity_year_z", "ev_to_sales_year_z", "pb_year_z", "fcf_yield_year_z",
    "exp_implied_revenue_cagr", "exp_implied_terminal_ebit_margin", "exp_implied_incremental_roic",
    "exp_implied_reinvestment_rate", "exp_duration_risk", "exp_valuation_pressure_score",
    "exp_current_revenue_growth_3y", "exp_current_operating_margin", "exp_current_roic_proxy",
    "exp_cost_anchor", "omega_feasibility_score", "omega_downside_anchor_score",
]
BASE_CAT_FEATURES = ["omega_regime", "sector", "industry"]

GAP_NUMERIC_FEATURES = [
    "spine_pred", "uniform_pred",
    "predicted_revenue_cagr_3y", "predicted_operating_margin_3y", "predicted_roic_3y", "predicted_fcf_margin_3y",
    "predicted_growth_gap_3y", "predicted_margin_gap_3y", "predicted_roic_gap_3y", "predicted_fcf_delta_3y",
    "exp_implied_revenue_cagr", "exp_implied_terminal_ebit_margin", "exp_implied_incremental_roic",
    "exp_implied_reinvestment_rate", "exp_duration_risk", "exp_valuation_pressure_score",
    "omega_feasibility_score", "omega_downside_anchor_score",
    "fcf_yield", "debt_assets", "revenue_growth_3y", "roic_proxy", "operating_margin",
]


def feature_cols_available(frame, cols):
    return [c for c in cols if c in frame.columns]


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=20)
    except TypeError:
        try:
            return OneHotEncoder(handle_unknown="ignore", sparse=False, min_frequency=20)
        except TypeError:
            return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(frame, numeric_features, categorical_features):
    numeric_features = feature_cols_available(frame, numeric_features)
    categorical_features = feature_cols_available(frame, categorical_features)
    transformers = []
    if numeric_features:
        transformers.append(("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features))
    if categorical_features:
        transformers.append(("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]), categorical_features))
    if not transformers:
        raise ValueError("No available features.")
    return ColumnTransformer(transformers=transformers, remainder="drop"), numeric_features, categorical_features


def fundamental_model(seed):
    return HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.035,
        max_iter=240,
        max_leaf_nodes=22,
        min_samples_leaf=22,
        l2_regularization=0.12,
        random_state=seed,
    )


def fit_future_fundamental_models(core):
    models = {}
    # Important: build a fresh ColumnTransformer for each target. Reusing the same
    # preprocessor object across pipelines lets later one-hot fits mutate earlier
    # pipelines and can trigger n_features mismatch during rolling folds.
    _, numeric_features, categorical_features = make_preprocessor(core, BASE_NUMERIC_FEATURES, BASE_CAT_FEATURES)
    feature_list = numeric_features + categorical_features
    for i, target in enumerate(FUNDAMENTAL_TARGETS):
        train = core.dropna(subset=[target]).copy()
        if len(train) < 600:
            continue
        target_preprocessor, _, _ = make_preprocessor(train, BASE_NUMERIC_FEATURES, BASE_CAT_FEATURES)
        pipe = Pipeline([
            ("prep", target_preprocessor),
            ("model", fundamental_model(SEED + i * 17)),
        ])
        pipe.fit(train[feature_list], train[target].clip(-1.0, 1.0))
        models[target] = {"pipeline": pipe, "features": feature_list, "rows": int(len(train))}

    # Fold-local smoke test: catches preprocessor/model feature drift immediately.
    if models:
        smoke = core[feature_list].head(min(25, len(core))).copy()
        for target, model_info in models.items():
            _ = model_info["pipeline"].predict(smoke[model_info["features"]])
    return models


def predict_future_fundamentals(models, frame):
    out = frame.copy()
    defaults = {
        "future_revenue_cagr_3y": out.get("revenue_growth_3y", pd.Series(np.nan, index=out.index)),
        "future_operating_margin_3y": out.get("operating_margin", pd.Series(np.nan, index=out.index)),
        "future_roic_3y": out.get("roic_proxy", pd.Series(np.nan, index=out.index)),
        "future_fcf_margin_3y": out.get("fcf_margin", pd.Series(np.nan, index=out.index)),
    }
    for target in FUNDAMENTAL_TARGETS:
        pred_col = target.replace("future_", "predicted_")
        if target in models:
            model_info = models[target]
            out[pred_col] = model_info["pipeline"].predict(out[model_info["features"]])
        else:
            out[pred_col] = defaults[target]
    return out


def add_gap_features(frame):
    out = frame.copy()
    out["predicted_growth_gap_3y"] = out["predicted_revenue_cagr_3y"] - out["exp_implied_revenue_cagr"]
    out["predicted_margin_gap_3y"] = out["predicted_operating_margin_3y"] - out["exp_implied_terminal_ebit_margin"]
    out["predicted_roic_gap_3y"] = out["predicted_roic_3y"] - out["exp_implied_incremental_roic"]
    out["predicted_fcf_delta_3y"] = out["predicted_fcf_margin_3y"] - out["fcf_margin"]
    out["economic_gap_formula_pred"] = (
        out["spine_pred"]
        + 0.45 * out["predicted_growth_gap_3y"].clip(-0.30, 0.30)
        + 0.25 * out["predicted_margin_gap_3y"].clip(-0.30, 0.30)
        + 0.15 * out["predicted_roic_gap_3y"].clip(-0.35, 0.35)
        + 0.15 * out["predicted_fcf_delta_3y"].clip(-0.25, 0.25)
        - 0.06 * out["exp_duration_risk"].fillna(0).clip(0, 1)
    ).clip(-0.35, 0.55)
    return out

## 6. Prepare Mature Dataset


In [ ]:
data = mask_immature_forward_returns(featured)
data = add_future_fundamental_targets(data)

for c in [TARGET, "year"] + FUNDAMENTAL_TARGETS:
    if c in data.columns:
        data[c] = pd.to_numeric(data[c], errors="coerce")
data["year"] = data["year"].astype("Int64")

ACTIVE_3Y_LENSES = [name for name in LENS_NAMES if name != "capitalCycle" and f"pred_{name}" in data.columns]
lens_cols = [f"pred_{name}" for name in ACTIVE_3Y_LENSES]
for c in lens_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")

data = data.dropna(subset=["ticker", "year", TARGET] + lens_cols).copy()
data["year"] = data["year"].astype(int)

mature_counts = data.groupby("year").size().rename("rows").reset_index()
future_counts = data[FUNDAMENTAL_TARGETS].notna().sum().rename("non_null_future_target").reset_index().rename(columns={"index": "target"})
print("Mature rows:", len(data), "tickers:", data["ticker"].nunique())
print("Active lenses:", ACTIVE_3Y_LENSES)
display(mature_counts)
display(future_counts)

## 7. Purged Economic Gap Fold Runner


In [ ]:
def run_economic_gap_fold(frame, val_year):
    # With a 3Y target, labels from year T are only known around T+3.
    # Tune years are Y-5 and Y-4, so the latest tune label matures in Y-1.
    tune_years = [val_year - HORIZON_YEARS - 2, val_year - HORIZON_YEARS - 1]
    core_end = min(tune_years) - 1
    latest_tune_label_year = max(tune_years) + HORIZON_YEARS
    if latest_tune_label_year >= val_year:
        raise AssertionError(f"Tune labels leak into validation year {val_year}: latest label year {latest_tune_label_year}")

    core = frame[frame["year"] <= core_end].copy()
    tune = frame[frame["year"].isin(tune_years)].copy()
    val = frame[frame["year"] == val_year].copy()
    if len(core) < MIN_CORE_ROWS or len(tune) < MIN_TUNE_ROWS or len(val) < MIN_VAL_ROWS:
        return None, None, {
            "val_year": val_year,
            "skip_reason": "insufficient_rows",
            "core_end": core_end,
            "tune_years": tune_years,
            "latest_tune_label_year": latest_tune_label_year,
            "core_rows": len(core),
            "tune_rows": len(tune),
            "val_rows": len(val),
        }

    spine_weights = fit_simplex_spine(core, lens_cols)
    for fold_frame in [core, tune, val]:
        fold_frame["spine_pred"] = apply_spine(fold_frame, lens_cols, spine_weights)
        fold_frame["uniform_pred"] = fold_frame[lens_cols].mean(axis=1)

    fundamental_models = fit_future_fundamental_models(core)
    tune_gap = add_gap_features(predict_future_fundamentals(fundamental_models, tune))
    val_gap = add_gap_features(predict_future_fundamentals(fundamental_models, val))

    gap_features = feature_cols_available(tune_gap, GAP_NUMERIC_FEATURES)
    residual_train = tune_gap.dropna(subset=[TARGET] + gap_features).copy()
    head_kind = "huber"
    if len(residual_train) < 120:
        head_kind = "formula_only"
        val_gap["economic_gap_adjustment"] = val_gap["economic_gap_formula_pred"] - val_gap["spine_pred"]
        val_gap["economic_gap_pred"] = val_gap["economic_gap_formula_pred"]
    else:
        y_resid = (residual_train[TARGET] - residual_train["spine_pred"]).clip(-0.35, 0.35)
        try:
            head = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", HuberRegressor(alpha=0.006, epsilon=1.35, max_iter=1200)),
            ])
            head.fit(residual_train[gap_features], y_resid)
        except Exception:
            head_kind = "ridge"
            head = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("model", Ridge(alpha=4.0)),
            ])
            head.fit(residual_train[gap_features], y_resid)

        adjustment = pd.Series(head.predict(val_gap[gap_features]), index=val_gap.index).clip(-0.20, 0.20)
        formula_adjustment = (val_gap["economic_gap_formula_pred"] - val_gap["spine_pred"]).clip(-0.20, 0.20)
        # Small blend: robust learned residual remains anchored to the explicit economic-gap formula.
        val_gap["economic_gap_adjustment"] = (0.70 * adjustment + 0.30 * formula_adjustment).clip(-0.20, 0.20)
        val_gap["economic_gap_pred"] = (val_gap["spine_pred"] + val_gap["economic_gap_adjustment"]).clip(-0.35, 0.55)

    single_maes = {name: mae_np(val_gap[f"pred_{name}"], val_gap[TARGET]) for name in ACTIVE_3Y_LENSES}
    best_single = min(single_maes, key=single_maes.get)
    fold = {
        "val_year": int(val_year),
        "core_end": int(core_end),
        "tune_years": ",".join(map(str, tune_years)),
        "latest_tune_label_year": int(latest_tune_label_year),
        "purged_selection_clean": bool(latest_tune_label_year < val_year),
        "core_rows": int(len(core)),
        "tune_rows": int(len(tune)),
        "val_rows": int(len(val_gap)),
        "head_kind": head_kind,
        "fundamental_models": {k: v["rows"] for k, v in fundamental_models.items()},
        "economic_gap_mae": mae_np(val_gap["economic_gap_pred"], val_gap[TARGET]),
        "formula_mae": mae_np(val_gap["economic_gap_formula_pred"], val_gap[TARGET]),
        "spine_mae": mae_np(val_gap["spine_pred"], val_gap[TARGET]),
        "uniform_mae": mae_np(val_gap["uniform_pred"], val_gap[TARGET]),
        "best_single": best_single,
        "best_single_mae": float(single_maes[best_single]),
        "economic_gap_ic": ic_np(val_gap["economic_gap_pred"], val_gap[TARGET]),
        "formula_ic": ic_np(val_gap["economic_gap_formula_pred"], val_gap[TARGET]),
        "spine_ic": ic_np(val_gap["spine_pred"], val_gap[TARGET]),
        "uniform_ic": ic_np(val_gap["uniform_pred"], val_gap[TARGET]),
        "economic_gap_decile_spread": decile_spread(val_gap, "economic_gap_pred"),
        "formula_decile_spread": decile_spread(val_gap, "economic_gap_formula_pred"),
        "spine_decile_spread": decile_spread(val_gap, "spine_pred"),
    }
    fold["beats_spine_mae"] = bool(fold["economic_gap_mae"] < fold["spine_mae"])
    fold["beats_uniform_mae"] = bool(fold["economic_gap_mae"] < fold["uniform_mae"])
    fold["beats_best_single_mae"] = bool(fold["economic_gap_mae"] < fold["best_single_mae"])
    fold["positive_ic"] = bool(fold["economic_gap_ic"] > 0)
    fold["positive_decile"] = bool(fold["economic_gap_decile_spread"] > 0)

    keep_cols = [
        "ticker", "year", "company_name", "sector", "industry", "omega_regime", TARGET,
        "spine_pred", "uniform_pred", "economic_gap_pred", "economic_gap_formula_pred", "economic_gap_adjustment",
        "predicted_growth_gap_3y", "predicted_margin_gap_3y", "predicted_roic_gap_3y", "predicted_fcf_delta_3y",
        "exp_implied_revenue_cagr", "exp_implied_terminal_ebit_margin", "exp_implied_incremental_roic",
        "exp_duration_risk", "omega_feasibility_score", "omega_downside_anchor_score",
    ] + lens_cols
    preds = val_gap[[c for c in keep_cols if c in val_gap.columns]].copy()
    preds["val_year"] = int(val_year)
    return fold, preds, None

## 8. Execute Purged Rolling-Origin Validation


In [ ]:
folds = []
prediction_frames = []
skipped = []
start = time.time()
for val_year in ROLLING_VAL_YEARS:
    print(f"Running economic-gap fold {val_year}...")
    fold, preds, aux = run_economic_gap_fold(data, val_year)
    if fold is None:
        skipped.append(aux)
        print("  skipped", aux)
        continue
    folds.append(fold)
    prediction_frames.append(preds)
    print(" ", {k: fold[k] for k in [
        "val_year", "head_kind", "economic_gap_mae", "formula_mae", "spine_mae", "uniform_mae",
        "best_single_mae", "economic_gap_ic", "economic_gap_decile_spread"
    ]})

folds_df = pd.DataFrame(folds)
economic_gap_predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
print("Elapsed minutes:", round((time.time() - start) / 60, 2))
print("Skipped folds:", skipped)
if not folds_df.empty:
    print("Purged label check:", folds_df[["val_year", "core_end", "tune_years", "latest_tune_label_year", "purged_selection_clean"]].to_dict(orient="records"))
display(folds_df)

## 9. Aggregate Scorecard and Gates


In [ ]:
if folds_df.empty:
    raise RuntimeError("No economic-gap rolling folds completed.")

pooled = economic_gap_predictions.copy()
summary = {
    "version": NOTEBOOK_VERSION,
    "validation_protocol": "purged_annual_rolling_origin_train_le_y_minus_6_tune_y_minus_5_y_minus_4_validate_y",
    "model_family": "economic_gap_transition_model",
    "fold_count": int(len(folds_df)),
    "fold_years": [int(x) for x in folds_df["val_year"].tolist()],
    "total_val_rows": int(len(pooled)),
    "economic_gap_mae_pooled": mae_np(pooled["economic_gap_pred"], pooled[TARGET]),
    "formula_mae_pooled": mae_np(pooled["economic_gap_formula_pred"], pooled[TARGET]),
    "spine_mae_pooled": mae_np(pooled["spine_pred"], pooled[TARGET]),
    "uniform_mae_pooled": mae_np(pooled["uniform_pred"], pooled[TARGET]),
    "economic_gap_ic_pooled": ic_np(pooled["economic_gap_pred"], pooled[TARGET]),
    "formula_ic_pooled": ic_np(pooled["economic_gap_formula_pred"], pooled[TARGET]),
    "spine_ic_pooled": ic_np(pooled["spine_pred"], pooled[TARGET]),
    "uniform_ic_pooled": ic_np(pooled["uniform_pred"], pooled[TARGET]),
    "economic_gap_decile_pooled": decile_spread(pooled, "economic_gap_pred"),
    "formula_decile_pooled": decile_spread(pooled, "economic_gap_formula_pred"),
    "spine_decile_pooled": decile_spread(pooled, "spine_pred"),
    "avg_economic_gap_mae": float(folds_df["economic_gap_mae"].mean()),
    "avg_formula_mae": float(folds_df["formula_mae"].mean()),
    "avg_spine_mae": float(folds_df["spine_mae"].mean()),
    "avg_uniform_mae": float(folds_df["uniform_mae"].mean()),
    "avg_economic_gap_ic": float(folds_df["economic_gap_ic"].mean()),
    "beats_spine_mae_share": float(folds_df["beats_spine_mae"].mean()),
    "beats_uniform_mae_share": float(folds_df["beats_uniform_mae"].mean()),
    "beats_best_single_mae_share": float(folds_df["beats_best_single_mae"].mean()),
    "positive_ic_share": float(folds_df["positive_ic"].mean()),
    "positive_decile_share": float(folds_df["positive_decile"].mean()),
    "head_kind_counts": folds_df["head_kind"].value_counts().to_dict(),
    "purged_selection_clean_share": float(folds_df["purged_selection_clean"].mean()),
    "filter_audit": filter_audit,
}
summary["mae_lift_vs_spine"] = summary["spine_mae_pooled"] - summary["economic_gap_mae_pooled"]
summary["mae_lift_vs_uniform"] = summary["uniform_mae_pooled"] - summary["economic_gap_mae_pooled"]
summary["mae_lift_vs_formula"] = summary["formula_mae_pooled"] - summary["economic_gap_mae_pooled"]

gates = {
    "enough_folds": summary["fold_count"] >= 6,
    "enough_rows": summary["total_val_rows"] >= 2500,
    "filter_product_canaries_clean": len(filter_audit["must_exclude_product_survivors"]) == 0,
    "filter_operating_canaries_kept": len(filter_audit["wrongly_removed_operating_canaries"]) == 0,
    "purged_selection_clean": summary["purged_selection_clean_share"] == 1.0,
    "pooled_beats_spine_mae_by_20bps": summary["mae_lift_vs_spine"] > 0.002,
    "pooled_beats_uniform_mae_by_20bps": summary["mae_lift_vs_uniform"] > 0.002,
    "beats_spine_in_70pct_folds": summary["beats_spine_mae_share"] >= 0.70,
    "beats_uniform_in_70pct_folds": summary["beats_uniform_mae_share"] >= 0.70,
    "beats_best_single_in_60pct_folds": summary["beats_best_single_mae_share"] >= 0.60,
    "positive_pooled_ic": summary["economic_gap_ic_pooled"] > 0.025,
    "positive_ic_in_60pct_folds": summary["positive_ic_share"] >= 0.60,
    "positive_pooled_decile": summary["economic_gap_decile_pooled"] > 0.030,
    "positive_decile_in_50pct_folds": summary["positive_decile_share"] >= 0.50,
}
production_candidate = all(gates.values())
summary["gates"] = gates
summary["production_candidate"] = bool(production_candidate)
summary["product_mode"] = "economic_gap_model_candidate" if production_candidate else "economic_gap_research_only"

print(json.dumps(summary, indent=2)[:16000])
display(pd.DataFrame([gates]).T.rename(columns={0: "passed"}))

## 10. Export Artifacts


In [ ]:
out_dir = ARTIFACT_ROOT / f"{ARTIFACT_NAME_PREFIX}_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M%S')}"
out_dir.mkdir(parents=True, exist_ok=True)

folds_df.to_csv(out_dir / "economic_gap_folds.csv", index=False)
economic_gap_predictions.to_csv(out_dir / "economic_gap_predictions.csv", index=False)
(out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
(out_dir / "filter_audit.json").write_text(json.dumps(filter_audit, indent=2), encoding="utf-8")
manifest = {
    "version": NOTEBOOK_VERSION,
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "artifact_dir": str(out_dir),
    "data_cutoff_date": DATA_CUTOFF_DATE,
    "summary": summary,
    "gates": gates,
    "decision": "PROMOTE_ECONOMIC_GAP_MODEL_CANDIDATE" if production_candidate else "KEEP_ECONOMIC_GAP_RESEARCH_ONLY",
}
(out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2)[:16000])
print("Artifact dir:", out_dir)